Imports

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


Paths

In [2]:
BASE = Path("..")  # لأنك داخل project/src
DATA_PATH = BASE / "data" / "clean_expenses.csv"
PLOTS_DIR = BASE / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH.resolve())
print("PLOTS_DIR:", PLOTS_DIR.resolve())


DATA_PATH: D:\courses\En + Ai + Github\Chat GPT course\Smart_Expenses_Project\project\data\clean_expenses.csv
PLOTS_DIR: D:\courses\En + Ai + Github\Chat GPT course\Smart_Expenses_Project\project\plots


Load Data

In [3]:
df = pd.read_csv(DATA_PATH)

# تحويل التاريخ
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# تنظيف category
df["category"] = df["category"].astype(str).str.strip()

# amount رقم
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

# حذف أي صفوف ناقصة
df = df.dropna(subset=["date", "amount", "category", "type"])

df.head()


,date,type,category,amount,payment_method,description,day_name,week,is_weekend,signed_amount
0,2026-01-01,income,Income,1200000,transfer,Monthly salary,Thursday,1,False,1200000
1,2026-01-01,expense,Bills,85000,cash,Internet subscription,Thursday,1,False,-85000
2,2026-01-02,expense,Food,18000,cash,Grocery items,Friday,1,False,-18000
3,2026-01-02,expense,Transport,5000,cash,Taxi ride,Friday,1,False,-5000
4,2026-01-03,expense,Shopping,25000,card,House supplies,Saturday,1,True,-25000


Filter Expenses Only

In [4]:
# نشتغل على المصروفات فقط
exp = df[df["type"].str.lower() == "expense"].copy()

print("Rows (all):", len(df))
print("Rows (expenses):", len(exp))
exp.head()


Rows (all): 30
Rows (expenses): 27


,date,type,category,amount,payment_method,description,day_name,week,is_weekend,signed_amount
1,2026-01-01,expense,Bills,85000,cash,Internet subscription,Thursday,1,False,-85000
2,2026-01-02,expense,Food,18000,cash,Grocery items,Friday,1,False,-18000
3,2026-01-02,expense,Transport,5000,cash,Taxi ride,Friday,1,False,-5000
4,2026-01-03,expense,Shopping,25000,card,House supplies,Saturday,1,True,-25000
5,2026-01-03,expense,Food,12000,cash,Breakfast,Saturday,1,True,-12000


Daily Spending Trend (Line) — Required

In [5]:
daily = exp.groupby(exp["date"].dt.date)["amount"].sum().sort_index()

plt.figure()
plt.plot(daily.index, daily.values)
plt.title("Daily Spending Trend")
plt.xlabel("Date")
plt.ylabel("Total Amount")
plt.xticks(rotation=45)
plt.tight_layout()

out_path = PLOTS_DIR / "daily_line.png"
plt.savefig(out_path)
plt.close()

print("Saved:", out_path)


Saved: ..\plots\daily_line.png


Spending by Category (Bar)

In [6]:
cat = exp.groupby("category")["amount"].sum().sort_values(ascending=False)

plt.figure()
plt.bar(cat.index, cat.values)
plt.title("Total Spending by Category")
plt.xlabel("Category")
plt.ylabel("Total Amount")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

out_path = PLOTS_DIR / "category_bar.png"
plt.savefig(out_path)
plt.close()

print("Saved:", out_path)


Saved: ..\plots\category_bar.png


Weekly Spending Trend (Line)

In [7]:
# تجميع أسبوعي اعتماداً على التاريخ (أوضح من week رقمية)
exp["week_period"] = exp["date"].dt.to_period("W").astype(str)
weekly = exp.groupby("week_period")["amount"].sum()

plt.figure()
plt.plot(weekly.index, weekly.values)
plt.title("Weekly Spending Trend")
plt.xlabel("Week")
plt.ylabel("Total Amount")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

out_path = PLOTS_DIR / "weekly_trend.png"
plt.savefig(out_path)
plt.close()

print("Saved:", out_path)


Saved: ..\plots\weekly_trend.png


Category Share (Pie)

In [8]:
# إذا تحب pie (اختياري)
cat = exp.groupby("category")["amount"].sum().sort_values(ascending=False)

plt.figure()
plt.pie(cat.values, labels=cat.index, autopct="%1.1f%%")
plt.title("Category Share")
plt.tight_layout()

out_path = PLOTS_DIR / "category_pie.png"
plt.savefig(out_path)
plt.close()

print("Saved:", out_path)


Saved: ..\plots\category_pie.png


Check Saved Plots

In [9]:
list(PLOTS_DIR.glob("*.png"))


[WindowsPath('../plots/category_bar.png'),
 WindowsPath('../plots/category_pie.png'),
 WindowsPath('../plots/daily_line.png'),
 WindowsPath('../plots/weekly_trend.png')]